<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-02-analyze-apparent-fishing-effort-argentinian-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze apparent fishing effort in Argentinian EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Analyze apparent fishing effort in [Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466) region and monitor industrial trawlers** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**, **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)**, and **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be installed easily from either the [Python Package Index (PyPI)](https://pypi.org/project/gfw-api-python-client/) or [Conda](https://anaconda.org/conda-forge/gfw-api-python-client)

In [1]:
# %pip install gfw-api-python-client

In [1]:
# %conda install -c conda-forge gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import datetime
import os

import pandas as pd

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: A Fisheries Enforcement Officer Monitoring Industrial Trawlers**

Maria, a fisheries enforcement officer in Argentina, monitors industrial trawlers operating within **[Argentinian Exclusive Economic Zone (EEZ)](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. His goal is to:

1. Analyzing apparent fishing effort for **trawlers** operating in Argentinian EEZ.
2. Identifying vessels involved in **apparent trawling activity** and determining their reported **flag states**.
3. Checking vessel history, including **potential transshipment** and **port visits**.
4. Generating reports to support fisheries enforcement decisions.

**APIs Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Retrieve **apparent fishing effort** data for trawlers.
2. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Group vessels by ID that are involved in **trawling activity**.
3. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** – Retrieve **vessel identity** & **ownership** details.
4. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** – Fetch **port visits** & **potential transshipment** history. 

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)
- [Vessel ID](https://globalfishingwatch.org/our-apis/documentation#vessel-id)
- [Vessel API - Vessel identity information](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Argentinian EEZ

Before making API requests, Maria must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Argentinian EEZ, the region ID is 8466](https://www.marineregions.org/gazetteer.php?p=details&id=8466)** (public-eez-areas dataset).

**Note:** See how to use the [Reference Data API - Usage Guides](https://globalfishingwatch.github.io/gfw-api-python-client/usage-guides/references-data-api.html) to obtain and filter predefined [**Regions of Interest (ROIs)**](https://globalfishingwatch.org/our-apis/documentation#regions), such as Exclusive Economic Zones (**EEZs**), Marine Protected Areas (**MPAs**), and Regional Fisheries Management Organizations (**RFMOs**).

In [5]:
eez_rois_result = await gfw_client.references.get_eez_regions(iso3="ARG")
arg_eez_roi = eez_rois_result.data()[0]

In [6]:
arg_eez_roi.id, arg_eez_roi.dataset, arg_eez_roi.label, arg_eez_roi.iso3

('8466', 'public-eez-areas', 'Argentinian Exclusive Economic Zone', 'ARG')

## Step 1: Retrieve Apparent Fishing Effort in Argentinian EEZ

Maria **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **apparent fishing effort for all vessels**, grouping them by **gear type** in **[Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - 8466 Argentinian EEZ
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 6 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Gear Type

**Why This Step?**

- Identifies which gear types (e.g., `trawlers`, `squid jiggers` etc.) are most active in the Argentinian EEZ.
- Establishes baseline fishing activity trends before narrowing the search to specific vessels.

In [7]:
end_date = datetime.date.today()

In [8]:
start_date = end_date - datetime.timedelta(weeks=24)

In [9]:
start_date, end_date

(datetime.date(2026, 1, 9), datetime.date(2026, 6, 26))

In [10]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="HIGH",
    group_by="GEARTYPE",
    temporal_resolution="MONTHLY",
    start_date=start_date,
    end_date=end_date,
    spatial_aggregation=True,
    region=arg_eez_roi,
)

In [11]:
step_1_report_df = step_1_report_result.df()

In [12]:
step_1_report_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     51 non-null     str    
 1   detections               0 non-null      object 
 2   flag                     0 non-null      object 
 3   gear_type                51 non-null     str    
 4   hours                    51 non-null     float64
 5   vessel_ids               51 non-null     int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign                0 non-null    

In [13]:
step_1_report_df[["gear_type", "hours", "vessel_ids"]].head()

,gear_type,hours,vessel_ids
0,inconclusive,334.308056,7
1,set_longlines,108.396111,7
2,pole_and_line,9.884444,1
3,inconclusive,137.272500,2
4,trawlers,40260.790000,229


In [14]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["gear_type"], as_index=False)
    .agg(hours=("hours", "sum"), vessel_ids=("vessel_ids", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [15]:
step_1_agg_report_df.head()

,gear_type,hours,vessel_ids
9,trawlers,216296.657778,1261
8,squid_jigger,51908.746667,339
1,fishing,8480.469444,74
2,fixed_gear,5068.472500,25
3,inconclusive,678.016944,17


### What We have Learned from Step 1

- Multiple **gear types** were potentially detected in Argentinian EEZ.
- `Trawlers` appear to be operating, but further **vessel-level investigation** is needed.

## Step 2: Retrieve Vessel IDs for Trawlers

Maria refines her **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** request to group by **vessel ID**, and filtering only for **trawlers** in **[Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - [8466 Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466)
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 6 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Vessel ID
4. **[Gear Type](https://globalfishingwatch.org/our-apis/documentation#gear-types-supported)** - Trawlers 

**Why Use group-by=VESSEL_ID?**

Grouping by **VESSEL_ID** allows **individual vessel identification** in the response. This is crucial for **tracking vessel activity** and, more importantly, linking each detected vessel to the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** in the next step. By structuring the query this way, we can fetch vessel details such as **flag, name, and ownership records** in **Step 3 below**.


In [16]:
step_2_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="HIGH",
    group_by="VESSEL_ID",
    temporal_resolution="ENTIRE",
    filters=["geartype in ('trawlers')"],
    start_date=start_date,
    end_date=end_date,
    spatial_aggregation=True,
    region=arg_eez_roi,
)

In [17]:
step_2_report_df = step_2_report_result.df()

In [18]:
step_2_report_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   date                     410 non-null    str                
 1   detections               0 non-null      object             
 2   flag                     410 non-null    str                
 3   gear_type                410 non-null    str                
 4   hours                    410 non-null    float64            
 5   vessel_ids               0 non-null      object             
 6   vessel_id                410 non-null    str                
 7   vessel_type              410 non-null    str                
 8   entry_timestamp          410 non-null    datetime64[us, UTC]
 9   exit_timestamp           410 non-null    datetime64[us, UTC]
 10  first_transmission_date  406 non-null    datetime64[us, UTC]
 11  last_transmission_date   406 non-null    da

In [19]:
step_2_report_df[["flag", "gear_type", "hours", "mmsi", "ship_name"]].head()

,flag,gear_type,hours,mmsi,ship_name
0,ARG,TRAWLERS,190.147500,701000714,EL MARISCO I
1,ARG,TRAWLERS,114.457500,701147000,HUAFENG828
2,CHN,TRAWLERS,32.674444,412549071,LU QING YUAN YU 225
3,ARG,TRAWLERS,454.279722,701006390,MADRE MARGARITA
4,ARG,TRAWLERS,1255.525278,701006050,SAN MATIAS


### Explore Vessels Potentially Engaged in Trawling Activity in the Argentinian EEZ

In [20]:
step_2_agg_report_df = (
    step_2_report_df.groupby(["flag", "gear_type", "mmsi", "ship_name"], as_index=False)
    .agg(hours=("hours", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [21]:
step_2_agg_report_df["hours"].describe()

count     396.000000
mean      546.203681
std       565.570234
min         0.102500
25%        82.493125
50%       375.651944
75%       834.973889
max      2647.745000
Name: hours, dtype: float64

In [22]:
step_2_agg_report_mask = step_2_agg_report_df["hours"] >= step_2_agg_report_df[
    "hours"
].quantile(0.99)

In [23]:
step_2_agg_report_df[step_2_agg_report_mask]

,flag,gear_type,mmsi,ship_name,hours
21,ARG,TRAWLERS,701000577,MISS TIDE,2647.745000
231,ARG,TRAWLERS,701006445,API V,2573.038889
281,ARG,TRAWLERS,701024000,ATLANTIC SURF III,2461.395000
285,ARG,TRAWLERS,701037000,DON PEDRO,2304.829444


### What We have Learned from Step 2

- There are vessels appear to have been engaged in potential **trawling activity** in Argentinian EEZ over the past 6 months.
- We will retrieve these vessels' `ownership`, `flag history`, and `authorizations` in **Step 3 to validate** them.

## Step 3: Retrieve Vessel Details Using the Vessels API

Maria queries the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** to **get detailed vessel identity and ownership records**. Please [learn more about Vessels API here](https://globalfishingwatch.org/our-apis/documentation#vessels-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information).

**Filters Used:**

1. **Vessel IDs** from [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api), **Step 2 above**.
2. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)** - `public-global-vessel-identity:latest`.
3. **[Includes](https://globalfishingwatch.org/our-apis/documentation#get-vessels-by-ids-url-parameters)** - `POTENTIAL_RELATED_SELF_REPORTED_INFO`.

**Note:** Vessels may change identifiers over time, such as their `Maritime Mobile Service Identity (MMSI)`,` International Maritime Organization (IMO) number)`, `call sign`, or even their `name`. These changes can occur due to `re-registration`, `changes in ownership`, or other `operational reasons` within the `AIS transponder`. Parameter (`includes = POTENTIAL_RELATED_SELF_REPORTED_INFO`) helps group all **vessel ids** that are **potentially related** as part of the **same physical vessel** based on publicly available registry information.

In [24]:
step_2_vessel_mmsis = list(step_2_agg_report_df[step_2_agg_report_mask]["mmsi"])

In [25]:
step_2_vessel_mmsis

['701000577', '701006445', '701024000', '701037000']

In [26]:
step_2_vessel_ids = list(
    step_2_report_df[step_2_report_df["mmsi"].isin(step_2_vessel_mmsis)][
        "vessel_id"
    ].unique()
)

In [27]:
step_2_vessel_ids

['4723e8576-6ec2-f4a7-2bc6-3bdb68f05a2a',
 '42b038c49-9432-46d4-042a-a730333e6510',
 '75184b8b0-0b20-6876-48e5-6582ba27ce50',
 '8e930bac5-594b-aa3f-081d-d12668819e1f']

In [28]:
step_3_vessels_result = await gfw_client.vessels.get_vessels_by_ids(
    ids=step_2_vessel_ids,
)

In [29]:
step_3_vessels_df = step_3_vessels_result.df()

In [30]:
step_3_vessels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         4 non-null      str   
 1   registry_info_total_records     4 non-null      int64 
 2   registry_info                   4 non-null      object
 3   registry_owners                 4 non-null      object
 4   registry_public_authorizations  4 non-null      object
 5   combined_sources_info           4 non-null      object
 6   self_reported_info              4 non-null      object
dtypes: int64(1), object(5), str(1)
memory usage: 356.0+ bytes


**Understanding Vessel Details Response Data**

- **registryInfoTotalRecords** – This represents the **number of registry records** found for the vessels.
- **registryInfo** – Contains **public registry data**. This data is sourced from official **vessel registries**.
- **registryOwners** – Lists the **registered owners** of the vessel based on public sources.
- **registryPublicAuthorizations** – Represents known **fishing authorizations** from public sources. Users should verify against national registries and RFMO records for additional context.
- **combinedSourcesInfo** – Provides inferred data from multiple sources, including. This is not explicitly reported by vessels but determined through **GFW's classification methods**.
- **selfReportedInfo** – Contains **AIS self-reported** data, including `MMSI`, `ship name`, and `flag` as broadcast by the **vessel itself**. Self-reported data may not always align with registry data and should be cross-checked.

In [31]:
step_3_vessels_df[["registry_info", "registry_owners", "self_reported_info"]]

,registry_info,registry_owners,self_reported_info
0,"[{'id': '45502524c9a150e77869ee647423dba1', 's...","[{'name': 'GLACIAR PESQUERA', 'flag': 'ARG', '...",[{'id': '8e930bac5-594b-aa3f-081d-d12668819e1f...
1,"[{'id': '0b811955df98f5dbf52e5076e8ab5645', 's...","[{'name': 'PEDRO MOSCUZZA HIJOS', 'flag': 'ARG...",[{'id': '4723e8576-6ec2-f4a7-2bc6-3bdb68f05a2a...
2,"[{'id': '582ac67871378cea17051ee3fc7c2821', 's...","[{'name': 'WANCHESE ARGENTINA', 'flag': 'ARG',...",[{'id': '75184b8b0-0b20-6876-48e5-6582ba27ce50...
3,"[{'id': 'ac9183ec6e744b635009f58d7a7e3d92', 's...","[{'name': 'IBERCONSA DE ARGENTINA', 'flag': 'A...",[{'id': '42b038c49-9432-46d4-042a-a730333e6510...


### Explore Vessels Registry Info

In [32]:
step_3_has_registry_info_mask = step_3_vessels_df[
    "registry_info"
].notna() & step_3_vessels_df["registry_info"].astype(bool)

In [33]:
step_3_registry_info_df = pd.json_normalize(
    step_3_vessels_df[step_3_has_registry_info_mask]["registry_info"].explode()
)

In [34]:
step_3_registry_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "gear_types", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,gear_types,source_code
0,701024000,ARG,ATLANTIC SURF III,ATLANTICSURF3,[TRAWLERS],"[IMO, SNP]"
1,701037000,ARG,DON PEDRO.,DONPEDRO,[TRAWLERS],"[IMO, SNP]"
2,701000577,ARG,MISS TIDE,MISSTIDE,[TRAWLERS],"[GFW-REVIEW, IMO]"
2,700000577,ARG,MISS TIDE,MISSTIDE,[TRAWLERS],"[GFW-REVIEW, IMO, SNP]"
3,701006445,ARG,API V,API5,[TRAWLERS],"[IMO, SNP]"


### Explore Registry Owners

In [35]:
step_3_has_registry_owners_mask = step_3_vessels_df[
    "registry_owners"
].notna() & step_3_vessels_df["registry_owners"].astype(bool)

In [36]:
step_3_registry_owners_df = pd.json_normalize(
    step_3_vessels_df[step_3_has_registry_owners_mask]["registry_owners"].explode()
)

In [37]:
step_3_registry_owners_match_registry_info_mask = step_3_registry_owners_df[
    "ssvid"
].isin(step_3_registry_info_df["ssvid"])

In [38]:
step_3_registry_owners_df[step_3_registry_owners_match_registry_info_mask][
    ["ssvid", "flag", "name", "source_code"]
]

,ssvid,flag,name,source_code
0,701024000,ARG,GLACIAR PESQUERA,"[SNP, IMO]"
1,701037000,ARG,PEDRO MOSCUZZA HIJOS,"[IMO, SNP]"
2,701000577,ARG,WANCHESE ARGENTINA,[IMO]
2,700000577,ARG,WANCHESE ARGENTINA,"[SNP, IMO]"
3,701006445,ARG,IBERCONSA DE ARGENTINA,"[IMO, SNP]"


### Explore Vessels Self Reported Info

In [39]:
step_3_has_self_reported_info_mask = step_3_vessels_df[
    "self_reported_info"
].notna() & step_3_vessels_df["self_reported_info"].astype(bool)

In [40]:
step_3_self_reported_info_df = pd.json_normalize(
    step_3_vessels_df[step_3_has_self_reported_info_mask][
        "self_reported_info"
    ].explode()
)

In [41]:
step_3_self_reported_info_match_registry_info_mask = step_3_self_reported_info_df[
    "ssvid"
].isin(step_3_registry_info_df["ssvid"])

In [42]:
step_3_self_reported_info_df[step_3_self_reported_info_match_registry_info_mask][
    ["ssvid", "flag", "ship_name", "n_ship_name", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,source_code
0,701024000,ARG,ATLANTIC SURF III,ATLANTICSURF3,[AIS]
1,701037000,ARG,DON PEDRO,DONPEDRO,[AIS]
1,701037000,ARG,DP,DP,[AIS]
1,701037000,ARG,DON PEDRO,DONPEDRO,[AIS]
2,701000577,ARG,MISS TIDE,MISSTIDE,[AIS]
2,700000577,NaN,MISS TIDE,MISSTIDE,[AIS]
2,701000577,ARG,MISS TIDE,MISSTIDE,[AIS]
3,701006445,ARG,API V,API5,[AIS]
3,701006445,ARG,API V,API5,[AIS]


### What We have Learned from Step 3

- **Vessel Identity:**
  - `MISS TIDE (mmsi: 701000577, flag: ARG)` - appears to be registered under Argentina (ARG)
  - `API V (mmsi: 701006445, flag: ARG)` - appears to be registered under Argentina (ARG)
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)` - appears to be registered under Argentina (ARG)
  - `DON PEDRO (mmsi: 701037000, flag: ARG)` - appears to be registered under Argentina (ARG)
- **Ownership & Historical Changes:**
  - `MISS TIDE (mmsi: 701000577, flag: ARG)` - **WANCHESE ARGENTINA** appears to be listed as the registered owner.
  - `API V (mmsi: 701006445, flag: ARG)` - **IBERCONSA DE ARGENTINA** appears to be listed as the registered owner.
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)` - **GLACIAR PESQUERA** appears to be listed as the registered owner.
  - `DON PEDRO (mmsi: 701037000, flag: ARG)` - **PEDRO MOSCUZZA HIJOS** appears to be listed as the registered owner.

## Step 4: Detect Potential Port Visits, Encounters, or Fishing Events

Now Maria checks `port visits`, `encounters`, and `fishing events` using the **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**, which allows **monitoring of vessel activities** such as `potential transshipments`, `unauthorized port entries`, or `fishing activity patterns`.

**Filters Used:**

1. **Vessel ID** from 4Wings API
2. **[Event Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - Port visits, encounters (potential transshipment), and fishing events.
3. **Time Range** - Last 6 months.
4. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)**:
   - `public-global-port-visits-events::latest` (Port Visits)
   - `public-global-encounters-events:latest` (Encounters between vessels)
   - `public-global-fishing-events:latest` (Fishing activity)
5. **[Encounter Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - CARRIER-FISHING

In [43]:
step_4_events_result = await gfw_client.events.get_all_events(
    datasets=[
        "public-global-encounters-events:latest",
        "public-global-fishing-events:latest",
        "public-global-port-visits-events:latest",
    ],
    vessels=step_2_vessel_ids,
    types=["ENCOUNTER", "FISHING", "PORT_VISIT"],
    start_date=start_date,
    end_date=end_date,
    encounter_types=["CARRIER-FISHING"],
    sort="-start",
)

In [44]:
step_4_events_df = step_4_events_result.df()

In [45]:
step_4_events_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 929 entries, 0 to 928
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         929 non-null    datetime64[us, UTC]
 1   end           929 non-null    datetime64[us, UTC]
 2   id            929 non-null    str                
 3   type          929 non-null    str                
 4   position      929 non-null    object             
 5   regions       929 non-null    object             
 6   bounding_box  929 non-null    object             
 7   distances     929 non-null    object             
 8   vessel        929 non-null    object             
 9   encounter     0 non-null      object             
 10  fishing       908 non-null    object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    21 non-null     object             
dtypes: datetime64[us, UTC

In [46]:
step_4_events_df["type"].value_counts()

type
fishing       908
port_visit     21
Name: count, dtype: int64

### Explore Apparent Fishing Events

In [47]:
step_4_fishing_events_df = step_4_events_df[step_4_events_df["fishing"].notna()]

In [48]:
step_4_fishing_df = pd.concat(
    [
        pd.json_normalize(step_4_fishing_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_fishing_events_df["fishing"], sep="_"),
    ],
    axis=1,
)

In [49]:
step_4_fishing_df.info()

<class 'pandas.DataFrame'>
Index: 908 entries, 10 to 927
Data columns (total 12 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id                                  908 non-null    str    
 1   name                                908 non-null    str    
 2   ssvid                               908 non-null    str    
 3   flag                                908 non-null    str    
 4   type                                908 non-null    str    
 5   public_authorizations               908 non-null    object 
 6   nextPort                            0 non-null      object 
 7   total_distance_km                   908 non-null    float64
 8   average_speed_knots                 908 non-null    float64
 9   average_duration_hours              0 non-null      object 
 10  potential_risk                      908 non-null    bool   
 11  vessel_public_authorization_status  908 non-null    str    


In [50]:
step_4_fishing_df[
    [
        "name",
        "ssvid",
        "total_distance_km",
        "average_speed_knots",
    ]
]

,name,ssvid,total_distance_km,average_speed_knots
10,API V,701006445,21.281764,4.159375
11,API V,701006445,24.251514,4.293750
12,API V,701006445,59.370875,4.660937
13,API V,701006445,8.297152,4.800000
14,API V,701006445,49.013410,3.661111
...,...,...,...,...
923,ATLANTIC SURF III,701024000,81.924980,4.230000
924,ATLANTIC SURF III,701024000,279.915578,4.318182
925,ATLANTIC SURF III,701024000,7.111741,4.537500
926,ATLANTIC SURF III,701024000,11.170620,4.703226


In [51]:
step_4_fishing_df["ssvid"].value_counts()

ssvid
701024000    297
701037000    261
701000577    215
701006445    135
Name: count, dtype: int64

### Explore Port Visit Events

In [52]:
step_4_port_visit_events_df = step_4_events_df[step_4_events_df["port_visit"].notna()]

In [53]:
step_4_port_visits_df = pd.concat(
    [
        pd.json_normalize(step_4_port_visit_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_port_visit_events_df["port_visit"], sep="_"),
    ],
    axis=1,
)

In [54]:
step_4_port_visits_df.info()

<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 928
Data columns (total 37 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   id                                             21 non-null     str    
 1   name                                           21 non-null     str    
 2   ssvid                                          21 non-null     str    
 3   flag                                           21 non-null     str    
 4   type                                           21 non-null     str    
 5   public_authorizations                          21 non-null     object 
 6   nextPort                                       0 non-null      object 
 7   visit_id                                       21 non-null     str    
 8   confidence                                     21 non-null     str    
 9   duration_hrs                                   21 non-null     float64


In [55]:
step_4_port_visits_df[
    [
        "name",
        "ssvid",
        "confidence",
        "start_anchorage_name",
        "intermediate_anchorage_name",
        "end_anchorage_name",
    ]
]

,name,ssvid,confidence,start_anchorage_name,intermediate_anchorage_name,end_anchorage_name
0,DON PEDRO,701037000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
1,DON PEDRO,701037000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
2,MISS TIDE,701000577,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
3,ATLANTIC SURF III,701024000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
4,ATLANTIC SURF III,701024000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
5,MISS TIDE,701000577,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
6,MISS TIDE,701000577,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
7,MISS TIDE,701000577,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
8,MISS TIDE,701000577,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
9,MISS TIDE,701000577,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA


In [56]:
step_4_port_visits_df["ssvid"].value_counts()

ssvid
701000577    8
701006445    6
701024000    5
701037000    2
Name: count, dtype: int64

### What We have learned from  step 4

- **Apparent Fishing Events:**
  - `MISS TIDE (mmsi: 701000577, flag: ARG)` - has been detected in multiple apparent fishing events over the past 6 months.
  - `API V (mmsi: 701006445, flag: ARG)` - has been detected in multiple apparent fishing events over the past 6 months.
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)` - has been detected in multiple apparent fishing events over the past 6 months.
  - `DON PEDRO (mmsi: 701037000, flag: ARG)` - has been detected in multiple apparent fishing events over the past 6 months.
- **Port Visit Events:**
  - `MISS TIDE (mmsi: 701000577, flag: ARG)`- potentially made multiple port visits, including stops at `MAR DEL PLATA`
  - `API V (mmsi: 701006445, flag: ARG)`- potentially made multiple port visits, including stops at `PUERTO DESEADO`
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)`- potentially made multiple port visits, including stops at `MAR DEL PLATA`
  - `DON PEDRO (mmsi: 701037000, flag: ARG)` - potentially made multiple port visits, including stops at `MAR DEL PLATA`
- **ENCOUNTER Events:** No explicit **ENCOUNTER** events were returned in the response dataset. Check more details [here](https://globalfishingwatch.org/faqs/what-is-a-vessel-encounter/). You can read more about transshipment behavior from our [report](https://globalfishingwatch.org/wp-content/uploads/GlobalViewOfTransshipment_Aug2017.pdf) or [scientific publication](https://www.frontiersin.org/articles/10.3389/fmars.2018.00240/full).

**Potential Considerations:**

- The vessel's fishing activities appear near the EEZ boundary, requiring further assessment of compliance with national or RFMO regulations.
- The absence of matching public authorizations in the RFMO registry does not necessarily indicate illegality, but it suggests that authorities may need to verify through national databases or official sources.

### Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Retrieve apparent fishing effort for **trawlers** within Argentinian EEZ.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** - Fetch **vessel identity**, **ownership history**, and **public authorizations**.
3. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** - Detect potential **port visits**, **encounters**, and **apparent fishing** events to analyze operational patterns.
4. **Assess potential risks** - Compare registry records, AIS data, and inferred vessel activity for enforcement follow-ups.
5. **Generate a report** - Provide a structured analysis for relevant authorities.